## LocustVR data preparation
This notebook requires that the data has been read out and preprocessed using "data_exploration_tutorial.ipynb" Session 1.0 using "locustvr_extractor.py" script.

### 0. Initialisation

#### 0.1 Define directory names and paths
- adjust name of utility directory containing "useful_tools and "data_cleaning"
- resolve_parent_directories is 0 when utility directory is in the same folder as this notebook, 1 when it is one folder above (Chi-Yu set 0)
- adjust root directory to /AG_Couzin-Fuchs/DATA
- decide to save as pickle or not (recommended for faster loading (if needed can be changed to save as hdf5))
- set save path



In [ ]:
utilities_name = "utilities"
resolve_parent_directories=0
root_dir_data = "/Volumes/DATA-1/experiment_trackball_Optomotor/locustVR"
save_pickle = True
save_path = "/Users/jonny/Desktop/Uni/Master/collevtive behavior/VR_analysis/collor_choice"

#### 0.2 Import packages

In [ ]:
import sys
from pathlib import Path

utilities_dir = Path("/Users/jonny/Desktop/Uni/Master/collevtive behavior/Locust analysis/utilities")
utilities_path = str(utilities_dir)
if utilities_path not in sys.path:
    sys.path.insert(0, utilities_path)


In [ ]:
import os,json
import sys
from pathlib import Path

utilities_dir = Path("/Users/jonny/Desktop/Uni/Master/collevtive behavior/Locust analysis/utilities")
if not (utilities_dir / 'useful_tools.py').is_file():
    raise FileNotFoundError(f'Could not find useful_tools.py in {utilities_dir}')
utilities_path = str(utilities_dir)
if utilities_path not in sys.path:
    sys.path.insert(0, utilities_path)

import pandas as pd
import matplotlib.pyplot as plt
plt.style.use('default')
import importlib
from LocustVR_data_analysis import utils_local

importlib.reload(utils_local)
from LocustVR_data_analysis.utils_local import align_and_flip_heading, reindex_t_by_state_transition, align_trajectories, \
    compute_directness_and_direction, flip_symmetric_states

from useful_tools import select_animals_gpt,find_file,column_name_list,get_fill_between_range,read_seq_config
from data_cleaning import findLongestConseqSubseq,interp_fill

json_file = "./analysis_methods_dictionary.json"

with open(json_file, "r") as f:
    analysis_methods = json.loads(f.read())

exp_name = analysis_methods.get("experiment_name")
variable_name = 'condition_code'

TRIAL_LABEL_PATTERN = (
    r'^(?:T\d+_)?CD(?P<distance_cm>\d+)_CS(?P<condition_code>\d+)_'
    r'S(?P<label_state>\d+)_L(?P<left_texture>[^_]+)_'
    r'R(?P<right_texture>[^_]+)_BT(?P<bait_texture>[^_]+)$'
)
LABEL_COLUMNS = [
    'distance_cm', 'condition_code', 'label_state',
    'left_texture', 'right_texture', 'bait_texture'
]

def add_trial_label_columns(data):
    """Parse the extended LocustVR label while keeping trial_label unchanged."""
    if 'trial_label' not in data.columns:
        raise KeyError("The input file does not contain a 'trial_label' column.")

    result = data.copy()
    extracted = result['trial_label'].astype(str).str.extract(TRIAL_LABEL_PATTERN)
    invalid_labels = result.loc[extracted.isna().all(axis=1), 'trial_label'].dropna().unique()
    if len(invalid_labels) > 0:
        examples = ', '.join(map(str, invalid_labels[:5]))
        raise ValueError(f'Unrecognised trial label(s), for example: {examples}')

    for column in ['distance_cm', 'condition_code', 'label_state']:
        extracted[column] = pd.to_numeric(extracted[column], errors='coerce').astype('Int64')

    for column in LABEL_COLUMNS:
        if column in result.columns:
            result[column] = result[column].fillna(extracted[column])
        else:
            result[column] = extracted[column]

    return result

### 1. Data preparation

#### 1.1 Load valid data
Finds directories containing HDF5 files. If individual selection is enabled in the configuration, the Google Sheet can still be used; otherwise the folders are discovered locally. The next cell deliberately keeps only the last eight sorted folders.

In [ ]:
# Define the path to your Excel file
dir_list = []
file_type=".h5"
using_google_sheet=True
sheet_name = 'LocustVR'
if analysis_methods.get("load_individual_data") == True:
    if using_google_sheet==True:
        # database_id = "1UL4eEUrQMapx9xz11-IyOSlPBcep3I9vBJ2uGgVudb8"
        #         #https://docs.google.com/spreadsheets/d/1UL4eEUrQMapx9xz11-IyOSlPBcep3I9vBJ2uGgVudb8/edit?usp=sharing
        # url = f"https://docs.google.com/spreadsheets/d/{database_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"
        database_id = "1UL4eEUrQMapx9xz11-IyOSlPBcep1I9vBJ2uGgVudb8"
                #https://docs.google.com/spreadsheets/d/1UL4eEUrQMapx9xz11-IyOSlPBcep1I9vBJ2uGgVudb8/edit?usp=sharing
        url = f"https://docs.google.com/spreadsheets/d/{database_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"
        #df = pd.read_excel(url, engine='openpyxl')## use this function if the file is not google sheet but uploaded excel file
        df = pd.read_csv(url)
    else:
        excel_file_path = "Z:/DATA/experiment_trackball_Optomotor/Locusts Management.xlsx"
        print(f"using a database {excel_file_path} from the server but this file might be outdated")
        # Create a 'with' statement to open and read the Excel file
        with pd.ExcelFile(excel_file_path) as xls:
            # Read the Excel sheet into a DataFrame with the sheet name (folder name)
            df = pd.read_excel(xls, sheet_name)
        ##list up the conditions and answers as strings for input argument to select animal. One condition must pair with one answer
    if analysis_methods.get("select_animals_by_condition") == True:
        animal_of_interest=select_animals_gpt(df,"Excluding this animal from analysis (Usually when animals die or molt, T/F)","F")
    else:
        animal_of_interest=df

    ID_array = animal_of_interest["ID"].values
    print(f"Valid animals IDs: \n {ID_array}")
    
    dir_list = [
    root.replace("\\", "/")
    for root, _, files in os.walk(root_dir_data)
    if any(ID in root for ID in ID_array)
    and any(file.endswith(file_type) for file in files)]
else:
    for root, dirs, files in os.walk(root_dir_data):
        for folder in dirs:
            folder_path=os.path.join(root,folder)
            if any(name.endswith(file_type) for name in os.listdir(folder_path)):
                dir_list.append(folder_path.replace("\\", "/"))
dir_list.sort()
print("Directories:")
for i in dir_list:
    print(i)


In [ ]:
dir_lis_2026 = dir_list[-8:] 
#dir_list = dir_list[-6:] 
dir_lis_2026


#### 1.2 Process data
Input: `XY_full.h5`. The complete `trial_label` is retained and additionally split into `distance_cm`, `condition_code`, `label_state`, `left_texture`, `right_texture`, and `bait_texture`.


In [ ]:
file_name = "/XY_full.h5"
pd_list = []
for index, this_dir in enumerate(dir_lis_2026):
    print(index, this_dir)
    this_pd = pd.read_hdf(this_dir + file_name)
    this_pd['animal_id'] = index
    this_pd = reindex_t_by_state_transition(this_pd, transition_from=0, transition_to=(1, 2))
    this_pd = align_and_flip_heading(this_pd)
    this_pd = add_trial_label_columns(this_pd)
    this_pd = align_trajectories(this_pd)
    this_pd = flip_symmetric_states(this_pd)
    this_pd = compute_directness_and_direction(this_pd)
    pd_list.append(this_pd)

if not pd_list:
    raise ValueError('No XY_full.h5 files were loaded from the selected directories.')

df = pd.concat(pd_list, ignore_index=True)

label_preview = (
    df[['trial_label'] + LABEL_COLUMNS]
    .drop_duplicates()
    .sort_values(['condition_code', 'label_state', 'left_texture', 'right_texture'])
)
display(label_preview.head(20))

if save_pickle:
    os.makedirs(save_path, exist_ok=True)
    df.to_pickle(os.path.join(save_path, 'locustvr_data.pkl'))

### 2. Load data

In [ ]:
df = pd.read_pickle(os.path.join(save_path, 'locustvr_data.pkl'))


In [ ]:
df.head(20)


In [ ]:
grouped = df.groupby("animal_id")["trial_id"].agg(["max", "nunique"])
print(grouped)